# scielo_pybot — demostración
Ejecute desde la carpeta del proyecto con el entorno virtual activo.

In [ ]:
import sys, os
os.chdir(os.path.abspath('..')) if os.path.basename(os.getcwd())=='notebooks' else None
sys.path.insert(0, os.getcwd())
from config.settings import Settings
s = Settings.load('config.json')
# Si config.json trae la ecuación vacía, escriba aquí la suya (o use esta de ejemplo):
if not s.equation.strip():
    s.equation = '(educ* OR ensino) AND ("inteligencia artificial" OR "inteligência artificial")'
s.equation[:80]

## 1. Motor booleano (offline)

In [ ]:
from core.boolean_engine import compile_query
q = compile_query(s.equation)
for t in ['Inteligencia artificial en la educación','Inteligência artificial no ensino','Inteligencia artificial en cardiología']:
    print(q.matches(t), '|', t)

## 2. Parser JATS con fixture **sintético** (offline)

In [ ]:
from core.jats_parser import parse_jats
p = parse_jats(open('tests/fixtures/synthetic_jats_fixture.xml','rb').read())
print(p['title'], p['doi'])
[(x['category'], x['title']) for x in p['sections']]

## 3. PRIMERA PRUEBA con artículo real (requiere internet)

In [ ]:
import subprocess
print(subprocess.run([sys.executable,'main.py','probe','--collection','per'],capture_output=True,text=True).stdout)

## 4. Búsqueda pequeña + diagnóstico

In [ ]:
from core.pipeline import run_search
s.max_results=5; s.collections=['per']; s.max_records_per_collection=50
matches, diag, repo = run_search(s)
print(diag.format())
for r in matches: print(r.collection, r.pid, r.year, r.title)

## 5. Exportación

In [ ]:
for f in ('csv','xlsx','json'): print(repo.export(f, s.exports_dir))
repo.to_dataframe().head()